<a href="https://colab.research.google.com/github/Jules-Vatel/SSI_SPRING/blob/main/SSI_SPRING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
url = "https://raw.githubusercontent.com/Jules-Vatel/SSI_SPRING/refs/heads/main/Offer-Westort_April%2028%2C%202026_08.08.csv"
df = pd.read_csv(url)


In [ ]:
#function for seven point party Id.
def seven_pt_pid(row):
  party = row['Q-PartyAffiliation']
  strength = row['Q-PolarAffiliation']
  lean = row['Q-MiddleAffiliation']
  if (party=='Democrat'):
      return 1 if strength == 'Strong' else 2
  elif(party=='Republican'):
      return 7 if strength == 'Strong' else 6
  elif(party in ['Independent','No Preference','Other Party (Please Specify)']):
      if (lean=='Closer to Democratic Party'):
          return 3
      elif (lean=='Closer to Republican Party'):
          return 5
      elif(lean=='Neither'):
          return 4
  return np.nan


In [ ]:
# function that gets party for leaners too
def get_party(row):
    pa = row['Q-PartyAffiliation']
    if(pa=='Democrat'):
        return 'Democrat'
    elif(pa=='Republican'):
        return 'Republican'
    elif(pa in ['Independent','No Preference','Other Party (Please Specify)']):
        ma = row.get('Q-MiddleAffiliation', np.nan)
        if(ma=='Closer to Democratic Party'):
            return 'Democrat'
        elif(ma=='Closer to Republican Party'):
            return 'Republican'
    return np.nan


In [ ]:
#tuple used to map the questions evaluating how comfortable respondents were to certain policies,
#relates to the social distance,close freinds distance, and inlaw distance questions,
#used to quantize each of the values

comfort_map = {
    'Very uncomfortable': 1, 'Uncomfortable': 2, 'Somewhat uncomfortable': 3,
    'Neither comfortable nor uncomfortable': 4, 'Somewhat comfortable': 5,
    'Comfortable': 6, 'Very comfortable': 7, 'Very Comfortable': 7
}

#same thing as comfort map but relates to the relection question so it can be mapped to varaibles
reelect_map = {
    'Very unlikely': 1, 'Unlikely': 2, 'Somewhat unlikely': 3,
    'Neither likely nor unlikely': 4, 'Somewhat likely': 5,
    'Likely': 6, 'Very likely': 7
}


#we can change these if we feel necessary

In [ ]:

#made a class to navigate the data better
#have access to each individual model,a 2d array containing all the models in a 2d array
# the class also has funciton that print the results from the regression direclty, I will add plotting functions and some other functions
#parameters to the constructor are a comfort_map for the social distance ,close freinds distance, inlaw distace questions and
# the relect map for the Q-ReElelection collumn
#affective polarization measured by first applying the comfort map to the collumns in question, these are added to a filler column with a _n added to the name
#example row#1 has temporary row row#1_n. Then the mean score of the three is taken and subtracted by 8 to get the Post treatment outcome varialbe.

def run_ols(formula,data):
  return sm.OLS.from_formula(formula,data).fit(cov_type='HC1')

class DATA_ANALYSIS:
  def __init__(self,df=df,cmap=comfort_map,rmap=reelect_map):
    EQ_AP1 = ""
    df = df[-df['Treat_Party'].str.contains('ImportId',na=False)].copy()
    df['Feeling_ThermR'] = pd.to_numeric(df['Q-FeelingThermR_1'], errors='coerce')
    df['Feeling_ThermD'] = pd.to_numeric(df['Q-FeelingThermD_1'], errors='coerce')
    df['PA']=df.apply(seven_pt_pid,axis=1)
    df['Party'] = df.apply(get_party, axis=1)
    df['ThermInP']  = np.where(df['Party'] == 'Democrat', df['Feeling_ThermD'], df['Feeling_ThermR'])
    df['ThermOutP'] = np.where(df['Party'] == 'Democrat', df['Feeling_ThermR'], df['Feeling_ThermD'])
    # get pre-treatment affective polarization
    df['APpre'] = (df['ThermInP'] - df['ThermOutP']).abs()
    for col in ['Q-InLawDist', 'Q-CloseFriendDist', 'Q-Social Distance 3']:
      df[col + '_n'] = df[col].map(cmap)
    # get affective polarization after as additive] value
    df['APpost'] = df[['Q-InLawDist_n', 'Q-CloseFriendDist_n','Q-Social Distance 3_n']].mean(axis=1)
    df['TB'] = df['Q-ReElection'].map(rmap)
    df['T1'] = ((df['Treat_Party'] == 'InParty') & (df['Treat_Frame'] == 'Electoral')).astype(int)
    df['T2'] = ((df['Treat_Party'] == 'InParty') & (df['Treat_Frame'] == 'Democracy')).astype(int)
    df['T3'] = ((df['Treat_Party'] == 'InParty') & (df['Treat_Frame'] == 'Policy')).astype(int)
    df['T4'] = ((df['Treat_Party'] == 'OutParty') & (df['Treat_Frame'] == 'Electoral')).astype(int)
    df['T5'] = ((df['Treat_Party'] == 'OutParty') & (df['Treat_Frame'] == 'Democracy')).astype(int)
    df['T6'] = ((df['Treat_Party'] == 'OutParty') & (df['Treat_Frame'] == 'Policy')).astype(int)

    analysis_vars = ['APpost','TB','APpre']
    full = df.dropna(subset=analysis_vars).copy()
    AP = run_ols('APpost ~ T1 + T2 + T3 + T4 + T5 + T6 + APpre', full)
    TB = run_ols('TB ~ T1 + T2 + T3 + T4 + T5 + T6 + APpre', full)
    self.AP = AP
    self.TB = TB
    self.f1_AP=f1_AP = AP.f_test('T1=0, T2=0, T3=0, T4=0, T5=0, T6=0')
    self.f1_TB=f1_TB = TB.f_test('T1=0, T2=0, T3=0, T4=0, T5=0, T6=0')
    self.f2_AP=f2_AP = AP.f_test('T1 + T2 + T3 = T4 + T5 + T6')
    self.f2_TB=f2_TB = TB.f_test('T1 + T2 + T3 = T4 + T5 + T6')
    self.f3_AP_EP=f3_AP_EP = AP.f_test('T1 + T4 = T3 + T6')
    self.f3_AP_DP=f3_AP_DP=AP.f_test('T2 + T5 = T3 + T6')
    self.f3_AP_ED=f3_AP_ED = AP.f_test('T1 + T4 = T2 + T5')
    self.f3_TB_EP=f3_TB_EP = TB.f_test('T1 + T4 = T3 + T6')
    self.f3_TB_DP=f3_TB_DP = TB.f_test('T2 + T5 = T3 + T6')
    self.f3_TB_ED=f3_TB_ED = TB.f_test('T1 + T4 = T2 + T5')
    self.f4_AP=f4_AP = AP.f_test('T1 - T3 = T4 - T6, T2 - T3 = T5 - T6')
    self.f4_TB=f4_TB = TB.f_test('T1 - T3 = T4 - T6, T2 - T3 = T5 - T6')
    self.model=[AP,TB]
    self.ftests=[[f1_AP,f1_TB],[f2_AP,f2_TB],[f3_AP_EP,f3_TB_EP,f3_AP_DP,f3_TB_DP,f3_AP_ED,f3_TB_DP],[f4_AP,f4_TB]]
    self.samples_full = len(full)

  def print_summary_all(self):
    labels=["Affective Polarization","Tolarance for Backsliding"]
    model = self.model
    for i in range(len(model)):
      print(f"Model Results ({labels[i]})")
      print(model[i].summary(),"\n")

  def print_summary_single(self,id):
    model = self.model

    n = None
    labels=["Affective Polarization","Tolarance for Backsliding"]
    if(id.upper()=="AP"):
      n = 0
    else:
      n = 1
    print(f"Results ({labels[n]})")
    print(model[n].summary(),"\n")

  def print_f_tests_all(self):
    ftests = self.ftests
    labels = [["RQ1: Any treatment (AP)","RQ1: Any treatment (TB)"],
      ["RQ2: In vs Out party (AP)","RQ2: In vs Out party (TB)"],
      ["RQ3: Electoral vs Policy (AP)","RQ3: Electoral vs Policy (TB)","RQ3: Democracy vs Policy (AP)","RQ3: Democracy vs Policy (TB)","RQ3: Electoral vs Democracy (AP)","RQ3: Electoral vs Democracy (TB)"],
      ["RQ4: Framing x Party (AP)","RQ4: Framing x Party (TB)"]]
    for i in range(len(ftests)):
      for j in range(len(ftests[i])):
        print(f"{labels[i][j]} {ftests[i][j]}")

    def print_f_tests_single(self,R_Question,id,inter_id=None):
      pass
      #will make this function aswell if needed at all


    def graph_delta_ap(self):
      models = self.models()
      fig,ax = plt.subplots()
      x = ["Model #1","Model #2","Model #3","Model #4"]
      y = []
      #plotting function not complete yet
      pass














In [ ]:

results = DATA_ANALYSIS()
#results.print_summary_single("AP")
results.print_summary_all()





Model Results (Affective Polarization)
                            OLS Regression Results                            
Dep. Variable:                 APpost   R-squared:                       0.070
Model:                            OLS   Adj. R-squared:                 -0.003
Method:                 Least Squares   F-statistic:                     1.249
Date:                Fri, 01 May 2026   Prob (F-statistic):              0.285
Time:                        22:19:05   Log-Likelihood:                -124.83
No. Observations:                  97   AIC:                             265.7
Df Residuals:                      89   BIC:                             286.3
Df Model:                           7                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    

In [ ]:
results.print_f_tests_all()

RQ1: Any treatment (AP) <F test: F=1.4563484077822846, p=0.20244102418577, df_denom=89, df_num=6>
RQ1: Any treatment (TB) <F test: F=0.561311101899831, p=0.7599634983328105, df_denom=89, df_num=6>
RQ2: In vs Out party (AP) <F test: F=1.6261415678133582, p=0.20555631023709262, df_denom=89, df_num=1>
RQ2: In vs Out party (TB) <F test: F=0.9493033905211404, p=0.3325374488703394, df_denom=89, df_num=1>
RQ3: Electoral vs Policy (AP) <F test: F=0.937003934127865, p=0.33567271645875185, df_denom=89, df_num=1>
RQ3: Electoral vs Policy (TB) <F test: F=0.1637485040697852, p=0.6866994491487108, df_denom=89, df_num=1>
RQ3: Democracy vs Policy (AP) <F test: F=4.156804692340096, p=0.044434363626344134, df_denom=89, df_num=1>
RQ3: Democracy vs Policy (TB) <F test: F=0.4755823897081343, p=0.49222436852300755, df_denom=89, df_num=1>
RQ3: Electoral vs Democracy (AP) <F test: F=1.5293789959299435, p=0.21945915053809575, df_denom=89, df_num=1>
RQ3: Electoral vs Democracy (TB) <F test: F=0.4755823897081343